# 最终状态系统雏形

**发射分布按维度选**：价格维度用 Student-t HMM，波动维度用 Gaussian HMM。

理由不是 NLL 更高，而是**应用指标**：价格维度 t 的风险分层更好（ES5 极差 1.94→2.56、
非重叠复核 3.7→6.3），波动维度 t 的分层塌了（9.83→2.98、6.57→3.27）。

看四张状态图 + 两套九宫格（时间线着色 / 3×3 热力图 / 逐年动画）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "state_layer").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from experiment.state_lab.matrix import _index_panel
from experiment.hmm_state.stage1 import build_groups, ORDER_COL
from experiment.hmm_state.final import fit_dim, grid, MODEL

pd.set_option("display.width", 160)
px.defaults.template = "plotly_white"

px_panel = _index_panel()
groups = build_groups(px_panel)

fitted = {}
for name, feats in groups.items():
    fitted[name] = fit_dim(name, feats, ORDER_COL[name])
    print(f"{name:6s} {fitted[name]['model']:14s} NLL={fitted[name]['nll']:.4f} "
          f"熵={fitted[name]['entropy']:.3f} 占比={fitted[name]['share']}")

短价格    Student-t(ν=5) NLL=1.2619 熵=0.079 占比={0: 0.507, 1: 0.258, 2: 0.235}


短波动    Gaussian       NLL=0.2915 熵=0.090 占比={0: 0.591, 1: 0.345, 2: 0.064}


长价格    Student-t(ν=5) NLL=2.1564 熵=0.064 占比={0: 0.495, 1: 0.231, 2: 0.274}


长波动    Gaussian       NLL=0.5853 熵=0.052 占比={0: 0.412, 1: 0.464, 2: 0.124}


In [2]:
PALETTE = ["#2E86C1", "#27AE60", "#C0392B", "#8E44AD", "#D68910", "#17A589"]
# 波动档分色系（蓝/橙/红），价格档调明暗
CELL_COLORS = ["#AED6F1", "#5499C7", "#1A5276",
               "#F9E79F", "#E67E22", "#A04000",
               "#F5B7B1", "#CB4335", "#641E16"]
CELL_ORDER = [f"{p}-{v}" for v in range(3) for p in range(3)]

## 工具函数（先跑）

In [3]:
def plot_states(name: str) -> go.Figure:
    r = fitted[name]
    states = r["states"]
    close = px_panel["close"].reindex(states.index)
    prof = r["profile"]

    fig = make_subplots(rows=2, cols=1, shared_xaxes=False, row_heights=[0.68, 0.32],
                        vertical_spacing=0.12,
                        subplot_titles=(f"{name}（{r['model']}）：价格按过滤状态着色",
                                        "状态画像（各特征均值）"))
    fig.add_trace(go.Scatter(x=close.index, y=close.values, mode="lines",
                             line=dict(color="#BDC3C7", width=1), name="价格",
                             showlegend=False), row=1, col=1)
    for i, s in enumerate(sorted(states.unique())):
        m = states == s
        seg = close.where(m)
        fig.add_trace(go.Scatter(x=seg.index, y=seg.values, mode="lines+markers",
                                 line=dict(color=PALETTE[i], width=2.2),
                                 marker=dict(size=4, color=PALETTE[i]),
                                 name=f"状态 {s}（{int(m.sum())} 天）"), row=1, col=1)
    for i, s in enumerate(sorted(prof.index)):
        fig.add_trace(go.Bar(x=prof.columns, y=prof.loc[s].values, name=f"状态 {s}",
                             marker_color=PALETTE[i], showlegend=False), row=2, col=1)
    fig.update_layout(height=620, margin=dict(l=40, r=20, t=60, b=40),
                      hovermode="x unified",
                      legend=dict(orientation="h", yanchor="bottom", y=1.02))
    fig.update_yaxes(title_text="价格", row=1, col=1)
    fig.update_yaxes(title_text="特征均值", row=2, col=1)
    return fig


def _cells(pk, vk) -> pd.Series:
    return grid(fitted[pk]["states"], fitted[vk]["states"])


def plot_grid(pk: str, vk: str, title: str) -> go.Figure:
    cell = _cells(pk, vk)
    close = px_panel["close"].reindex(cell.index)
    share = cell.value_counts(normalize=True).reindex(CELL_ORDER).fillna(0) * 100

    heat = np.zeros((3, 3))
    for c, v in share.items():
        p, vv = c.split("-")
        heat[int(vv), int(p)] = v

    fig = make_subplots(rows=2, cols=2, row_heights=[0.62, 0.38],
                        column_widths=[0.66, 0.34],
                        specs=[[{"colspan": 2}, None], [{}, {"type": "heatmap"}]],
                        subplot_titles=(f"{title}：价格按九宫格着色（"
                                        f"{fitted[pk]['model']} × {fitted[vk]['model']}）",
                                        "各格价格路径", "3×3 时间占比（%）"))
    fig.add_trace(go.Scatter(x=close.index, y=close.values, mode="lines",
                             line=dict(color="#D5D8DC", width=1), name="价格",
                             showlegend=False), row=1, col=1)
    for i, c in enumerate(CELL_ORDER):
        m = cell == c
        if not m.any():
            continue
        seg = close.where(m)
        fig.add_trace(go.Scatter(x=seg.index, y=seg.values, mode="lines+markers",
                                 line=dict(color=CELL_COLORS[i], width=2.5),
                                 marker=dict(size=4, color=CELL_COLORS[i]),
                                 name=f"{c}（{int(m.sum())} 天，{share[c]:.1f}%）"),
                      row=1, col=1)
    for i, c in enumerate(CELL_ORDER):
        sub = close[cell == c]
        if len(sub) < 5:
            continue
        fig.add_trace(go.Scatter(x=sub.index, y=sub.values, mode="lines",
                                 line=dict(color=CELL_COLORS[i], width=1),
                                 name=c, showlegend=False), row=2, col=1)
    fig.add_trace(go.Heatmap(z=heat, x=["价格0", "价格1", "价格2"],
                             y=["波动0", "波动1", "波动2"], text=np.round(heat, 1),
                             texttemplate="%{text}%", colorscale="Blues",
                             showscale=False), row=2, col=2)
    fig.update_layout(height=720, margin=dict(l=40, r=20, t=70, b=40),
                      hovermode="closest",
                      legend=dict(orientation="h", yanchor="bottom", y=1.06))
    return fig


def plot_animation(pk: str, vk: str, title: str):
    cell = _cells(pk, vk)
    d = pd.DataFrame({"格": cell})
    d["价格"] = d["格"].str.split("-").str[0].astype(int)
    d["波动"] = d["格"].str.split("-").str[1].astype(int)
    d["年份"] = d.index.year
    cnt = d.groupby(["年份", "价格", "波动"]).size().reset_index(name="天数")
    fig = px.scatter(cnt, x="价格", y="波动", size="天数", color="天数",
                     animation_frame="年份", size_max=45,
                     range_x=[-0.5, 2.5], range_y=[-0.5, 2.5],
                     color_continuous_scale="Blues",
                     title=f"{title}：3×3 占用随年份")
    fig.update_layout(height=460, margin=dict(l=40, r=20, t=60, b=40))
    fig.update_xaxes(tickmode="array", tickvals=[0, 1, 2],
                     ticktext=["价格0(弱)", "价格1(中)", "价格2(强)"])
    fig.update_yaxes(tickmode="array", tickvals=[0, 1, 2],
                     ticktext=["波动0(低)", "波动1(中)", "波动2(高)"])
    return fig

## 一、短价格（Student-t）

In [4]:
plot_states('短价格').show()

## 二、短波动（Gaussian）

In [5]:
plot_states('短波动').show()

## 三、长价格（Student-t）

In [6]:
plot_states('长价格').show()

## 四、长波动（Gaussian）

In [7]:
plot_states('长波动').show()

## 五、四个维度的画像与占比

In [8]:
for name, r in fitted.items():
    print(f"=== {name}（{r['model']}）NLL={r['nll']:.4f} 熵={r['entropy']:.3f} "
          f"最大后验={r['maxp']:.3f} ===")
    display(r["profile"])
    print("占比:", r["share"])
    print()

=== 短价格（Student-t(ν=5)）NLL=1.2619 熵=0.079 最大后验=0.967 ===


,ret_5,ret_10,er_10,snr_10
0,-1.40,-2.83,0.33,-0.82
1,0.86,1.21,0.15,0.38
2,2.24,5.02,0.54,1.50


占比: {0: 0.507, 1: 0.258, 2: 0.235}

=== 短波动（Gaussian）NLL=0.2915 熵=0.090 最大后验=0.965 ===


,rv_5,atr_5,gk_10,abs_ret_5
0,12.91,1.16,10.63,0.67
1,20.48,1.84,17.03,1.05
2,36.28,3.20,31.36,2.03


占比: {0: 0.591, 1: 0.345, 2: 0.064}

=== 长价格（Student-t(ν=5)）NLL=2.1564 熵=0.064 最大后验=0.973 ===


,ret_20,ret_60,er_20,er_60,snr_20
0,-4.12,-4.45,0.24,0.12,-0.83
1,1.93,0.67,0.11,0.10,0.40
2,6.40,10.94,0.35,0.19,1.28


占比: {0: 0.495, 1: 0.231, 2: 0.274}

=== 长波动（Gaussian）NLL=0.5853 熵=0.052 最大后验=0.981 ===


,gk_20,atr_20,rv_40,vol_rank_250
0,10.49,1.13,13.66,0.23
1,14.75,1.60,19.46,0.66
2,25.72,2.56,32.33,0.91


占比: {0: 0.412, 1: 0.464, 2: 0.124}



## 六、短周期九宫格（短价格 × 短波动）

In [9]:
plot_grid('短价格', '短波动', '短周期').show()

In [10]:
plot_animation('短价格', '短波动', '短周期').show()

## 七、中长期九宫格（长价格 × 长波动）

In [11]:
plot_grid('长价格', '长波动', '中长期').show()

In [12]:
plot_animation('长价格', '长波动', '中长期').show()

## 八、两套九宫格的风险画像

In [13]:
from experiment.hmm_state.stage1 import risk_by, nonoverlap_check

for label, (pk, vk) in {"短周期": ("短价格", "短波动"),
                        "中长期": ("长价格", "长波动")}.items():
    cell = _cells(pk, vk)
    print(f"=== {label} ===")
    display(risk_by(cell, px_panel))
    print("非重叠复核:", nonoverlap_check(cell, px_panel))
    print()

=== 短周期 ===


,n,占比%,未来均值%,ES5,ES5_CI,MAE中位%,MDD中位%,MDD_P90%,可用
格,,,,,,,,,
0-0,302,29.3,-0.45,-9.95,"[-11.3, -7.6]",-4.24,-4.92,-10.97,True
0-1,182,17.7,0.45,-9.42,"[-11.6, -6.8]",-3.60,-4.34,-10.46,True
0-2,33,3.2,6.20,-1.67,样本不足,-2.27,-2.96,-6.11,True
1-0,155,15.1,-0.92,-11.83,"[-13.5, -8.4]",-3.91,-4.25,-8.42,True
1-1,105,10.2,-0.44,-12.72,"[-14.3, -6.0]",-3.31,-4.76,-9.37,True
1-2,6,0.6,4.60,1.79,样本不足,-0.35,-5.70,-6.60,False
2-0,164,15.9,0.50,-7.92,"[-8.4, -6.2]",-2.37,-4.22,-6.73,True
2-1,59,5.7,-0.27,-11.49,"[-11.6, -9.8]",-4.04,-4.72,-11.12,True
2-2,23,2.2,4.64,-1.69,样本不足,-1.62,-4.06,-10.39,False


非重叠复核: {'样本': 52, '格数': 2, 'ES5极差': 4.9}

=== 中长期 ===


,n,占比%,未来均值%,ES5,ES5_CI,MAE中位%,MDD中位%,MDD_P90%,可用
格,,,,,,,,,
0-0,220,22.1,0.96,-9.21,"[-11.4, -5.7]",-3.34,-4.11,-10.97,True
0-1,216,21.7,0.47,-10.96,"[-11.8, -8.1]",-3.98,-4.39,-11.34,True
0-2,47,4.7,4.38,-2.20,"[-2.5, 0.8]",-1.03,-2.96,-4.81,True
1-0,88,8.9,-1.46,-10.56,"[-13.3, -7.5]",-4.39,-4.50,-8.40,True
1-1,125,12.6,-1.74,-13.30,"[-14.3, -8.0]",-4.51,-5.22,-11.86,True
1-2,15,1.5,-0.04,-6.13,样本不足,-1.79,-4.72,-6.06,False
2-0,110,11.1,1.96,-7.15,"[-8.7, -1.9]",-1.92,-3.99,-5.44,True
2-1,120,12.1,-1.31,-9.57,"[-10.4, -7.6]",-3.18,-5.05,-8.60,True
2-2,53,5.3,1.00,-5.00,"[-5.3, -3.5]",-3.25,-5.98,-7.80,True


非重叠复核: {'样本': 50, '格数': 2, 'ES5极差': 4.7}

